## Context

This notebook accompanies User Guide section 3 (Quick Start).  It runs a three-step plan, `load_dataset` then `data_summary` then `train_model`, through the governed gateway against the prebuilt fixture.  The point is to show the canonical setup pattern condensed onto one screen.  It does not rebuild the fixture — that is the job of `scripts/build_fixture.py`.

In [ ]:
# Phase 1B preamble: bootstrap runtime + load shared helpers
# Bootstrap: portable repo-root locator — walks up from CWD looking for
# knowlytix/harness/governance/tutorials/notebooks/_shared.py. Works in
# both the source clone (knowlytix/GMS) and any installed-wheel context
# (notebooks ship inside knowlytix-harness as of the 2026-05-07 move).
import os  # noqa: F401  (some notebooks use os.environ / os.makedirs)
import sys
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

_marker = Path('knowlytix') / 'harness' / 'governance' / 'tutorials' / 'notebooks' / '_shared.py'
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / _marker).exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

from knowlytix.harness.governance.tutorials.notebooks import _helpers_phase1 as h1
h1.bootstrap_runtime()  # stubs modeva if absent
from knowlytix.harness.governance.tutorials.notebooks import _shared as ug
import torch
DEVICE = ug.DEVICE
print(f'Device: {DEVICE}, CUDA available: {torch.cuda.is_available()}')


### Concept: the canonical setup pattern

Three objects matter at startup: a `GMSExpertStore`, a `SessionState`, and a `GovernedToolGateway` wired to both.  The store carries the geometric memory loaded from disk.  The session is the mutable scratchpad that tools write into and read from across calls.  The gateway is the single point through which all tool execution flows; it owns the three gates, the transcript, and (optionally) a fault-injection profile.

A `theta_plausibility` of 1.5 is the default published in the User Guide.  Calls whose geodesic distance to the relation triple `(context, should_call, tool_name)` exceed that threshold are blocked.  When the triple is not in the store the gate fails open, which is what permits unfamiliar contexts in the early tutorials.  Production stores tighten this by ingesting an explicit policy graph; that is what notebooks 16 and 18 do.

We run a minimal three-step plan against the iris dataset and a `RandomForestClassifier`.  Each step is a single `gateway.execute` call.  The interesting observation is the structure of the transcript: every step yields a `GatewayResult` with the same shape, so downstream consumers (orchestrator, verifier, tracer) do not branch on tool identity.  That uniformity is what makes the architecture composable.

In [ ]:
from knowlytix.harness.governance.session import SessionState
from knowlytix.harness.governance.governed_gateway import GovernedToolGateway
from knowlytix.harness.governance.tools import TOOL_DEFINITIONS

store = ug.load_ds_practices_store()
session = SessionState()
gateway = GovernedToolGateway(
    session=session,
    store=store,
    theta_plausibility=1.5,
    tool_schemas=TOOL_DEFINITIONS,
    device=str(DEVICE),
)
print('Setup OK. theta =', gateway.theta_plausibility)


In [ ]:
plan = [
    ('load_dataset',
     {'name': 'iris', 'builtin': 'iris'},
     'data_loading'),
    ('data_summary',
     {'dataset': 'iris'},
     'exploratory_data_analysis'),
    ('train_model',
     {'dataset': 'iris',
      'model_type': 'RandomForestClassifier',
      'model_name': 'rf_iris',
      'params': {'n_estimators': 30, 'random_state': 0}},
     'model_training'),
]
step_results = []
for tool, args, ctx in plan:
    r = gateway.execute(tool, args, context=ctx)
    step_results.append((tool, r))
    flag = 'OK ' if not r.blocked else 'BLK'
    print(f'{flag}  {tool:14s} {r.execution_time_ms:7.1f} ms'
          f'  reason={r.reason!r}')


In [ ]:
# Visualize per-step latency and pass/block status
import matplotlib.pyplot as plt

tools = [t for t, _ in step_results]
lat = [r.execution_time_ms for _, r in step_results]
ok  = [0 if r.blocked else 1 for _, r in step_results]
colors = ['tab:green' if o else 'tab:red' for o in ok]

fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(10, 3.6))
ax_l.bar(tools, lat, color=colors)
ax_l.set_ylabel('latency (ms)')
ax_l.set_title('Per-step gateway latency')
for label in ax_l.get_xticklabels():
    label.set_rotation(20)
ax_r.bar(tools, ok, color=colors)
ax_r.set_ylim(0, 1.2)
ax_r.set_yticks([0, 1])
ax_r.set_yticklabels(['blocked', 'allowed'])
ax_r.set_title('Per-step gate verdict')
for label in ax_r.get_xticklabels():
    label.set_rotation(20)
fig.tight_layout()
plt.show()


### Demo: an allowed call and a blocked call

The plan above runs cleanly through all three gates. To see the schema gate refuse a malformed call, run two contrastive `gateway.execute` calls back to back — one well-formed, one missing a required parameter. Both go through the same gateway with the same `theta_plausibility`; the only difference is the arguments. The gateway short-circuits on the first failing gate, so the blocked call never reaches the policy or plausibility checks. For the policy- and plausibility-block paths (each with its own demo cell), see notebook 16.

In [ ]:
# (1) Allowed — load_dataset with the required 'name' parameter.
allowed = gateway.execute(
    'load_dataset',
    {'name': 'iris', 'builtin': 'iris'},
    context='data_loading',
)
print(f'allowed.blocked      = {allowed.blocked}')
print(f'allowed.reason       = {allowed.reason}')
print(f'allowed.gate_results = {allowed.gate_results}')
print(f'allowed.display      = {allowed.display}')

# (2) Blocked — train_model called with no arguments.
#     Schema gate rejects before the tool runs.
blocked = gateway.execute(
    'train_model',
    {},                       # missing dataset, model_type, model_name
    context='training',
)
print()
print(f'blocked.blocked      = {blocked.blocked}')
print(f'blocked.reason       = {blocked.reason}')
print(f'blocked.gate_results = {blocked.gate_results}')
print(f'blocked.display      = {blocked.display!r}')

# The blocked call short-circuited at gate 1. Policy and
# plausibility were never consulted, so gate_results carries
# only the single 'schema': False entry.
assert allowed.blocked is False
assert blocked.blocked is True
assert blocked.gate_results == {'schema': False}

### Exercises

1. Add a fourth step that calls `predict` on the `rf_iris` model.  Confirm it passes all three gates.
2. Lower `theta_plausibility` to 0.5 and re-run; every step should now block at the plausibility gate because no triple in `ds_practices_store` matches the synthetic context strings used here.
3. Inspect `gateway.transcript[-1]` and identify the field that an offline auditor would need to reconstruct the call.

### What you should now be able to do

- Set up a `GMSExpertStore`, `SessionState`,   and `GovernedToolGateway` in three lines.
- Run a multi-step plan as a list of   `(tool, args, context)` tuples.
- Read per-step `GatewayResult` to assert that no step   was blocked.
- Recognise that all steps go through the identical   triple-gate pipeline.

### Cross-references

- USER_GUIDE.md section 3 (Quick Start)
- `gms_agent_builder_ext/chapters/ch03.tex` for the   Quick Start as a chapter
- `knowlytix/harness/governance/orchestrator.py` for the   multi-step orchestration that NB 20 builds on top of   this primitive.

In [ ]:
# Final assertion: every step passed.
results_only = [r for _, r in step_results]
assert all(not r.blocked for r in results_only), (
    'a step was blocked: ' +
    str([(t, r.reason) for t, r in step_results if r.blocked])
)
print(f'OK: all {len(step_results)} steps passed the '
      f'three-gate pipeline.')
